In [1]:
import os,sys
os.chdir('../')
%pwd

'/Users/hh/UOB/FYP/arabic-sentiment-framework'

# PROCESSING SLSA DATASET

In [17]:
from pathlib import Path
import pandas as pd
import json

source_slsa = Path("raw_datasets/ar_reviews_100k.tsv")
df = pd.read_csv(source_slsa, sep="\t", dtype=str)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
shuffled_df = df.sample(n=len(df))

Shape: (99999, 2)
Columns: ['label', 'text']


In [23]:
shuffled_df = shuffled_df[0:30001]
shuffled_df['label'].value_counts()

label
Mixed       10079
Positive    10039
Negative     9883
Name: count, dtype: int64

In [24]:
TEXT_COL = "text"
LABEL_COL = "label"

In [25]:
from sklearn.model_selection import train_test_split
SEED = 42
TEST_RATIO = 0.10
DEV_RATIO = 0.10
train_df, test_df = train_test_split(
    shuffled_df,
    test_size=TEST_RATIO,
    random_state=SEED,
    stratify=shuffled_df[LABEL_COL]
)
dev_size_relative_to_train = DEV_RATIO / (1 - TEST_RATIO)
train_df, dev_df = train_test_split(
    train_df,
    test_size=dev_size_relative_to_train,
    random_state=SEED,
    stratify=train_df[LABEL_COL]
)


In [26]:
SMALL_N = 2400  # your small regime
train_small_df = train_df.sample(n=SMALL_N, random_state=SEED, replace=False)

train_big_df = train_df  # full train

print("Train small:", len(train_small_df))
print("Train big:", len(train_big_df))


Train small: 2400
Train big: 24000


In [ ]:

OUT_DIR = Path("artifacts/data/processed/slsa")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def to_jsonl(df, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            obj = {"text": row[TEXT_COL], "label": row[LABEL_COL]}
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

to_jsonl(train_small_df, Path.joinpath(OUT_DIR,"slsa_train_small.jsonl"))
to_jsonl(train_big_df, Path.joinpath(OUT_DIR, "slsa_train_full.jsonl"))
to_jsonl(dev_df, Path.joinpath(OUT_DIR, "slsa_dev.jsonl"))
to_jsonl(test_df, Path.joinpath(OUT_DIR, "slsa_test.jsonl"))

print("Saved to:", OUT_DIR.resolve())


Saved to: /Users/hh/UOB/FYP/arabic-sentiment-framework/artifacts/data/processed/slsa


In [29]:
a = train_small_df
print(a.iterrows())

<generator object DataFrame.iterrows at 0x13e5b8ba0>


In [32]:
import xml.etree.ElementTree as ET

def parse_sb2_review_level(xml_path: str):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []
    for review in root.findall(".//Review"):
        rid = review.get("rid")

        # Collect review text by concatenating all sentence texts
        sent_texts = []
        for s in review.findall(".//sentences/sentence"):
            t = s.find("text")
            if t is not None and t.text:
                sent_texts.append(t.text.strip())
        full_text = " ".join(sent_texts).strip()

        # Collect review-level opinions
        labels = []
        opinions_block = review.find("Opinions")
        if opinions_block is not None:
            for op in opinions_block.findall("Opinion"):
                labels.append({
                    "category": op.get("category"),
                    "polarity": op.get("polarity")
                })

        rows.append({
            "rid": rid,
            "text": full_text,
            "labels": labels
        })

    return rows


In [ ]:
import json
from pathlib import Path

def absa_rows_save_jsonl(rows, out_path: str):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


In [ ]:
def absa_train_dev_split(rows, dev_ratio=0.15):
    n = len(rows)
    n_dev = int(n * dev_ratio)
    dev = rows[:n_dev]        
    train = rows[n_dev:]     
    return train, dev


In [39]:
train_rows = parse_sb2_review_level("raw_datasets/SemEval2016_arabic/SemEval2016_arabic/AR_Hotels_Train_SB2.xml")
train_rows, dev_rows = split_train_dev(train_rows, dev_ratio=0.15)

save_jsonl(train_rows, "artifacts/data/processed/absa/absa_train_big.jsonl")
save_jsonl(dev_rows,   "artifacts/data/processed/absa/absa_dev.jsonl")

test_rows = parse_sb2_review_level("raw_datasets/SemEval2016_arabic/SemEval2016_arabic/AR_HOTE_SB2_TEST.xml.gold")
save_jsonl(test_rows, "artifacts/data/processed/absa/absa_test.jsonl")

